## **Read chunks**

In [ ]:
import json
path = './data'
with open(f'{path}/chunked_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

## **Semantic-based Retrieval**

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

2025-07-28 14:29:49.062089: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753712989.248647      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753712989.302104      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [3]:
import torch

def semantic_based_retrieval(query, chunks, model, top_k=10):
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    model.to(device)
    
    prefix_query = f"search_query: {query}"
    chunk_texts = [f"search_document: {chunk['chunk_text']}" for chunk in chunks]

    with torch.no_grad():        
        q_vector = model.encode(query, convert_to_tensor=True).unsqueeze(0)
        c_vectors = model.encode(chunk_texts, convert_to_tensor=True)

    q_vector = q_vector.to(device)
    c_vectors = c_vectors.to(device)
    cos = torch.nn.CosineSimilarity(dim=1)
    similarities = cos(q_vector, c_vectors)

    for i, chunk in enumerate(chunks):
        chunk['similarity'] = similarities[i].item()

    return sorted(chunks, key=lambda x: x['similarity'], reverse=True)[:top_k]

query = "In which part of New York City is the director of the romantic comedy 'Big Stone Gap' based?"
semantic_chunks = semantic_based_retrieval(query, data['chunks'], model, 10)
semantic_chunks

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3731 [00:00<?, ?it/s]

[{'chunk_id': '5a8e3ea95542995a26add48d_doc9_chunk0',
  'source_doc_title': 'Big Stone Gap (film)',
  'chunk_text': "Big Stone Gap is a 2014 American drama romantic comedy film written and directed by Adriana Trigiani and produced by Donna Gigliotti for Altar Identity Studios, a subsidiary of Media Society.  Based on Trigiani's 2000 best-selling novel of the same name, the story is set in the actual Virginia town of Big Stone Gap circa 1970s.  The film had its world premiere at the Virginia Film Festival on November 6, 2014.",
  'question_id': '5a8e3ea95542995a26add48d',
  'doc_index': 9,
  'chunk_index': 0,
  'similarity': 0.755217432975769},
 {'chunk_id': '5a877df15542993e715abf7a_doc9_chunk0',
  'source_doc_title': 'David Schwimmer',
  'chunk_text': 'David Lawrence Schwimmer (born November 2, 1966) is an American actor, director, and producer.  He was born in Flushing, Queens, New York, and his family moved to Los Angeles when he was 2.  He began his acting career performing in scho

## Save semantic chunks

In [ ]:
with open('./data/semantic_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(semantic_chunks, f)